This notebook attempts to run the test experiments from the previous DEV notebooks using code shifted to the `helpers` packages, to enable command line training (multiple experiments on single instance).

In [1]:
import sys
sys.path.insert(1, '../')
import helpers
import torch

In [4]:
resize_dim=224 #224

from torchvision import transforms
import torch
base_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.ToTensor(),
])

# Augmentation for contrastive learning
augment_transform = transforms.Compose([
    transforms.Resize((resize_dim, resize_dim)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

from torchvision.transforms import v2 as v2transforms
randomaugs = [
    v2transforms.ToImage(),
    v2transforms.Resize(size=[resize_dim*2]),  # Slightly larger for random crops
    v2transforms.RandomResizedCrop(resize_dim, scale=(0.6, 1.0), ratio=(0.8, 1.2)),
    v2transforms.RandomHorizontalFlip(p=0.5),
    v2transforms.RandomRotation(degrees=10),
    v2transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.1, hue=0.05),
    v2transforms.RandomApply([
        v2transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))
    ], p=0.3),
    v2transforms.RandomErasing(p=0.3, scale=(0.02, 0.15)),
    v2transforms.ToDtype(torch.float32, scale=True),  # Scale to [0,1] first
]

In [5]:
import os
dataset_path = "../split_node21"
process = "lung_seg" #"crop" "arch_seg"
bsz = 64
resize_dim = 224

# Load with default settings
train_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process, "train"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=augment_transform,
    cache_in_ram=True

)

# Load with default settings
test_dataloader = helpers.dataloading.load_image_pair_dataset(
    dataset_path=os.path.join(dataset_path, process, "test"),
    batch_size=bsz,
    crop_size=resize_dim,  # Standard ResNet50 input size
    symmetrical_transforms=True,
    class_to_idx={'nodule': 1, 'normal': 0},
    transform=base_transform,
    cache_in_ram=True
)

Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  2.64it/s]


Caching base images into RAM...


Caching: 100%|██████████| 5162/5162 [00:38<00:00, 135.30it/s]


Cached 5162 images
Total pairs: 2581
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 742
  normal (idx=0): 1839


Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  2.39it/s]


Caching base images into RAM...


Caching: 100%|██████████| 2214/2214 [00:20<00:00, 107.16it/s]

Cached 2214 images
Total pairs: 1107
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 318
  normal (idx=0): 789


In [6]:
def train_siamese_epoch(model, dataloader, criterion, optimizer, device):
    """Training loop for siamese network"""
    model.train()
    total_loss = 0
    
    for batch_idx, (img1, img2, labels, path1, path2) in enumerate(dataloader):
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)
        
        optimizer.zero_grad()
        emb1, emb2 = model(img1, img2)
        loss = criterion(emb1, emb2, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        # if batch_idx % 100 == 0:
        #     print(f'Batch {batch_idx}, Loss: {loss.item():.4f}')
    
    return total_loss / len(dataloader)


In [7]:
model_to_load = "rad"

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = helpers.models.SiameseNetwork(helpers.models.load_truncated_model(model_to_load), 
                       embedding_dim=128, freeze_backbone=True).to(device)
contrastive_loss = helpers.losses.ContrastiveLoss(margin=1.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [8]:
# radimagnet frozen backbone
for epoch in range(0,100):
    avg_loss = train_siamese_epoch(model, train_dataloader, contrastive_loss, optimizer, device)
    print(f"Epoch {epoch}, Average Loss: {avg_loss:.4f}")

Epoch 0, Average Loss: 0.2064
Epoch 1, Average Loss: 0.2066
Epoch 2, Average Loss: 0.2058
Epoch 3, Average Loss: 0.2068
Epoch 4, Average Loss: 0.2072
Epoch 5, Average Loss: 0.2051
Epoch 6, Average Loss: 0.2076
Epoch 7, Average Loss: 0.2075
Epoch 8, Average Loss: 0.2049
Epoch 9, Average Loss: 0.2041
Epoch 10, Average Loss: 0.2038
Epoch 11, Average Loss: 0.2027
Epoch 12, Average Loss: 0.2025
Epoch 13, Average Loss: 0.2013
Epoch 14, Average Loss: 0.2013
Epoch 15, Average Loss: 0.2022
Epoch 16, Average Loss: 0.2070
Epoch 17, Average Loss: 0.2027
Epoch 18, Average Loss: 0.2024
Epoch 19, Average Loss: 0.2009
Epoch 20, Average Loss: 0.2008
Epoch 21, Average Loss: 0.1996
Epoch 22, Average Loss: 0.2018
Epoch 23, Average Loss: 0.1983
Epoch 24, Average Loss: 0.2014
Epoch 25, Average Loss: 0.2018
Epoch 26, Average Loss: 0.2008
Epoch 27, Average Loss: 0.1989
Epoch 28, Average Loss: 0.2001
Epoch 29, Average Loss: 0.1991
Epoch 30, Average Loss: 0.2010
Epoch 31, Average Loss: 0.2013
Epoch 32, Average 